In [ ]:
import os
import sys

import torch

# Add project root directory to Python path to import modules from 'src'
sys.path.append(os.path.abspath(".."))

from src.dataset import load_aml_dataset
from src.model import GATv2
from src.utils import FocalLoss, compute_metrics

# ---------------------------------------------------------
# Step 1: Load Graph Dataset & Verify Tensor Dimensions
# ---------------------------------------------------------
print("=== Step 1: Testing Dataset Loader ===")
data_root = os.path.join("..", "data")
data = load_aml_dataset(root=data_root)

print(f"Total Nodes (Accounts): {data.num_nodes}")
print(f"Total Edges (Transactions): {data.num_edges}")
print(f"Node Features Matrix (x): {data.x.shape}")
print(f"Edge Index Tensor: {data.edge_index.shape}")
if data.edge_attr is not None:
    print(f"Edge Attributes Tensor: {data.edge_attr.shape}")
print(f"Class Distribution (Legitimate vs AML): {torch.bincount(data.y)}")

# ---------------------------------------------------------
# Step 2: Test Model Instantiation & Dry-Run Forward Pass
# ---------------------------------------------------------
print("\n=== Step 2: Testing Model Forward Pass ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate GATv2 model using graph dimensions
model = GATv2(
    in_channels=data.num_node_features,
    hidden_channels=64,
    out_channels=2,
    edge_dim=data.edge_attr.shape[1] if data.edge_attr is not None else 1,
    heads=4,
    dropout=0.2,
).to(device)

# Move graph data to active device (CPU/GPU)
data = data.to(device)

# Execute single forward pass in evaluation mode
model.eval()
with torch.no_grad():
    output_logits = model(data.x, data.edge_index, data.edge_attr)

print(f"Output Logits Shape: {output_logits.shape}")
assert output_logits.shape == (data.num_nodes, 2), "Error: Output shape mismatch!"
print("Forward pass successful: Dimension checks passed.")

# ---------------------------------------------------------
# Step 3: Sanity Check Loss Computation & Initial Metrics
# ---------------------------------------------------------
print("\n=== Step 3: Sanity Check Loss & Metrics ===")
criterion = FocalLoss(alpha=0.25, gamma=2.0)
initial_loss = criterion(output_logits, data.y)
print(f"Initial Untrained Loss: {initial_loss.item():.4f}")

# Extract predictions and calculate baseline metrics
predictions = output_logits.argmax(dim=-1).cpu().numpy()
probabilities = torch.softmax(output_logits, dim=-1).cpu().numpy()
metrics = compute_metrics(data.y.cpu().numpy(), predictions, probabilities)

print(f"Baseline F1-Score: {metrics['f1']:.4f}")
print(f"Baseline Recall:   {metrics['recall']:.4f}")